# Snippet from Contributing.md


In [ ]:
# There is no compitum.observers module (no EnergyObserver class exists) and
# no compitum.certificate module -- both are fictional. The real analogous
# test lives at tests/energy/test_symbolic_free_energy.py; this mirrors that
# real pattern instead.
import numpy as np
import pytest

from compitum.boundary import BoundaryAnalyzer
from compitum.capabilities import Capabilities
from compitum.coherence import CoherenceFunctional
from compitum.energy import SymbolicFreeEnergy
from compitum.metric import SymbolicManifoldMetric
from compitum.models import Model
from compitum.predictors import CalibratedPredictor


def test_energy_monotonic_wrt_distance():
    D = 6
    model = Model(
        name="m", center=np.zeros(D),
        capabilities=Capabilities(regions={"US"}, tools_allowed={"none"}), cost=0.1,
    )
    rng = np.random.default_rng(0)
    X = rng.standard_normal((128, D))
    predictors = {}
    for name in ("quality", "latency", "cost"):
        p = CalibratedPredictor()
        p.fit(X, np.full(128, 0.5))  # constant target isolates the distance effect
        predictors[name] = p
    metric = SymbolicManifoldMetric(D, rank=3, delta=1e-3)
    coherence = CoherenceFunctional(k=50)
    energy = SymbolicFreeEnergy(alpha=1.0, beta_t=0.5, beta_c=0.2, beta_d=0.3, beta_s=0.4)

    U_near, _, _ = energy.compute(np.zeros(D), model, predictors, coherence, metric)
    U_far, _, _ = energy.compute(np.ones(D) * 3.0, model, predictors, coherence, metric)

    assert U_near > U_far, "Utility should decrease with distance from the model center"
